# 01 - Ingesta Parquet RAW (Spark -> Snowflake)

Notebook de ingesta mensual para NYC Taxi Yellow/Green. El flujo es:

1. Leer configuracion desde variables de entorno.
2. Descargar/leer Parquet fuente.
3. Estandarizar columnas y tipos con Spark DataFrames.
4. Agregar metadatos de auditoria y clave natural `TRIP_NK`.
5. Validar calidad minima y deduplicar el lote.
6. Escribir a Snowflake RAW mediante tabla staging + `MERGE`.
7. Registrar auditoria por lote en `RAW.LOAD_AUDIT`.

La idempotencia se garantiza por `TRIP_NK`: si el mismo lote se ejecuta varias veces, Snowflake actualiza las filas existentes y solo inserta claves nuevas.


## 1. Configuracion desde variables de entorno

No se hardcodean credenciales, rutas ni rango temporal. Si falta una variable obligatoria, el notebook falla temprano para preservar reproducibilidad.


In [1]:
from collections import OrderedDict
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import urlretrieve
import datetime as dt
import glob
import os
import re

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


def log_step(message: str) -> None:
    timestamp = dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def require_env(name: str) -> str:
    value = os.environ.get(name)
    if value is None or value.strip() == '':
        raise ValueError(f'Missing required environment variable: {name}')
    return value.strip()


def parse_csv_env(name: str) -> list[str]:
    return [item.strip().lower() for item in require_env(name).split(',') if item.strip()]


def parse_months(name: str) -> list[int]:
    values = [int(item) for item in parse_csv_env(name)]
    invalid = [month for month in values if month < 1 or month > 12]
    if invalid:
        raise ValueError(f'{name} contains invalid months: {invalid}')
    return values


def clean_identifier(value: str, name: str = 'identifier') -> str:
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_$]*', value):
        raise ValueError(f'Invalid Snowflake {name}: {value!r}')
    return value.upper()


RUN_ID = require_env('RUN_ID')
SERVICES = parse_csv_env('SERVICES')
ALLOWED_SERVICES = {'yellow', 'green'}
unsupported_services = sorted(set(SERVICES) - ALLOWED_SERVICES)
if unsupported_services:
    raise ValueError(f'Unsupported SERVICES values: {unsupported_services}. Expected yellow and/or green.')

START_YEAR = int(require_env('START_YEAR'))
END_YEAR = int(require_env('END_YEAR'))
if START_YEAR > END_YEAR:
    raise ValueError('START_YEAR must be <= END_YEAR')

MONTHS = parse_months('MONTHS')
LOCAL_PARQUET_DIR = Path(require_env('LOCAL_PARQUET_DIR'))
LOCAL_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

RAW_SCHEMA = clean_identifier(require_env('SF_RAW_SCHEMA'), 'schema')
SF_OPTIONS = {
    'sfURL': require_env('SF_HOST'),
    'sfUser': require_env('SF_USER'),
    'sfPassword': require_env('SF_PASSWORD'),
    'sfDatabase': clean_identifier(require_env('SF_DATABASE'), 'database'),
    'sfSchema': RAW_SCHEMA,
    'sfWarehouse': clean_identifier(require_env('SF_WAREHOUSE'), 'warehouse'),
    'sfRole': clean_identifier(require_env('SF_ROLE'), 'role'),
}

BASE_URLS = {
    service: require_env(f'{service.upper()}_BASE_URL').rstrip('/')
    for service in SERVICES
}

log_step(
    'Configured run: '
    f'run_id={RUN_ID}, services={SERVICES}, years={START_YEAR}-{END_YEAR}, '
    f'months={MONTHS}, local_parquet_dir={LOCAL_PARQUET_DIR}, raw_schema={RAW_SCHEMA}'
)


[2026-04-07 22:14:41Z] Configured run: run_id=20260405_test_01, services=['green'], years=2020-2020, months=[1], local_parquet_dir=/tmp/nyc_taxi_tripdata, raw_schema=RAW


## 2. Sesion Spark y utilidades Snowflake

Se crea una sesion Spark en UTC. Las acciones contra Snowflake usan el conector Spark-Snowflake; no se usa `pandas` ni `write_pandas` para la carga principal.


In [2]:
def build_spark(app_name: str) -> SparkSession:
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
    )

    shuffle_partitions = os.environ.get('SPARK_SQL_SHUFFLE_PARTITIONS')
    if shuffle_partitions:
        builder = builder.config('spark.sql.shuffle.partitions', shuffle_partitions)

    driver_memory = os.environ.get('SPARK_DRIVER_MEMORY')
    if driver_memory:
        builder = builder.config('spark.driver.memory', driver_memory)

    executor_memory = os.environ.get('SPARK_EXECUTOR_MEMORY')
    if executor_memory:
        builder = builder.config('spark.executor.memory', executor_memory)

    jars_dirs = [os.environ.get('SPARK_JARS_DIR'), '/home/jovyan/jars', str(Path.cwd().parent / 'jars')]
    jar_paths = []
    for jars_dir in [path for path in jars_dirs if path]:
        jar_paths = sorted(glob.glob(str(Path(jars_dir) / '*.jar')))
        if jar_paths:
            log_step(f'Loading JARs from {jars_dir}: {jar_paths}')
            break

    if jar_paths:
        builder = (
            builder
            .config('spark.jars', ','.join(jar_paths))
            .config('spark.driver.extraClassPath', ':'.join(jar_paths))
            .config('spark.executor.extraClassPath', ':'.join(jar_paths))
        )
    else:
        log_step('No local Snowflake JARs found; relying on PYSPARK_SUBMIT_ARGS/SPARK_EXTRA_CLASSPATH if provided.')

    return builder.getOrCreate()


def assert_snowflake_connector(spark: SparkSession) -> None:
    class_names = [
        'net.snowflake.spark.snowflake.DefaultSource',
        'snowflake.DefaultSource',
    ]
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found. Check Snowflake Spark/JDBC JARs and restart the kernel after changing jar config.')


app = build_spark('01_ingesta_parquet_raw')
assert_snowflake_connector(app)

SF_UTILS = app._jvm.net.snowflake.spark.snowflake.Utils


def sf_run(sql: str) -> None:
    options = app._jvm.java.util.HashMap()
    for key, value in SF_OPTIONS.items():
        options.put(key, value)
    SF_UTILS.runQuery(options, sql)


def fq_table(table_name: str) -> str:
    return f'{RAW_SCHEMA}.{clean_identifier(table_name, "table")}'


[2026-04-07 22:14:41Z] Loading JARs from /home/jovyan/jars: ['/home/jovyan/jars/snowflake-jdbc-3.28.0.jar', '/home/jovyan/jars/spark-snowflake_2.12-3.1.8.jar']
[2026-04-07 22:14:45Z] Snowflake connector detected: net.snowflake.spark.snowflake.DefaultSource


## 3. Contrato RAW y auditoria

Las tablas RAW se crean si no existen. El contrato mantiene las columnas originales necesarias para los notebooks posteriores y agrega metadatos obligatorios de ingesta.


In [3]:
RAW_COLUMN_TYPES = OrderedDict([
    ('TRIP_NK', 'VARCHAR'),
    ('VENDORID', 'NUMBER(38,0)'),
    ('TPEP_PICKUP_DATETIME', 'TIMESTAMP_NTZ'),
    ('TPEP_DROPOFF_DATETIME', 'TIMESTAMP_NTZ'),
    ('LPEP_PICKUP_DATETIME', 'TIMESTAMP_NTZ'),
    ('LPEP_DROPOFF_DATETIME', 'TIMESTAMP_NTZ'),
    ('STORE_AND_FWD_FLAG', 'VARCHAR'),
    ('RATECODEID', 'NUMBER(38,0)'),
    ('PULOCATIONID', 'NUMBER(38,0)'),
    ('DOLOCATIONID', 'NUMBER(38,0)'),
    ('PASSENGER_COUNT', 'FLOAT'),
    ('TRIP_DISTANCE', 'FLOAT'),
    ('FARE_AMOUNT', 'FLOAT'),
    ('EXTRA', 'FLOAT'),
    ('MTA_TAX', 'FLOAT'),
    ('TIP_AMOUNT', 'FLOAT'),
    ('TOLLS_AMOUNT', 'FLOAT'),
    ('EHAIL_FEE', 'FLOAT'),
    ('IMPROVEMENT_SURCHARGE', 'FLOAT'),
    ('TOTAL_AMOUNT', 'FLOAT'),
    ('PAYMENT_TYPE', 'NUMBER(38,0)'),
    ('TRIP_TYPE', 'NUMBER(38,0)'),
    ('CONGESTION_SURCHARGE', 'FLOAT'),
    ('AIRPORT_FEE', 'FLOAT'),
    ('RUN_ID', 'VARCHAR'),
    ('SERVICE_TYPE', 'VARCHAR'),
    ('SOURCE_YEAR', 'NUMBER(4,0)'),
    ('SOURCE_MONTH', 'NUMBER(2,0)'),
    ('INGESTED_AT_UTC', 'TIMESTAMP_NTZ'),
    ('SOURCE_PATH', 'VARCHAR'),
])
RAW_COLUMNS = list(RAW_COLUMN_TYPES.keys())

AUDIT_COLUMN_TYPES = OrderedDict([
    ('RUN_ID', 'VARCHAR'),
    ('SERVICE', 'VARCHAR'),
    ('YEAR', 'NUMBER(4,0)'),
    ('MONTH', 'NUMBER(2,0)'),
    ('SOURCE_PATH', 'VARCHAR'),
    ('STATUS', 'VARCHAR'),
    ('ROWS_READ', 'NUMBER(38,0)'),
    ('ROWS_VALID', 'NUMBER(38,0)'),
    ('ROWS_REJECTED', 'NUMBER(38,0)'),
    ('ROWS_DEDUPED', 'NUMBER(38,0)'),
    ('ROWS_WRITTEN', 'NUMBER(38,0)'),
    ('BAD_KEY_ROWS', 'NUMBER(38,0)'),
    ('BAD_DISTANCE_ROWS', 'NUMBER(38,0)'),
    ('BAD_DURATION_ROWS', 'NUMBER(38,0)'),
    ('OUTSIDE_SOURCE_MONTH_ROWS', 'NUMBER(38,0)'),
    ('EVENT_TIMESTAMP_UTC', 'TIMESTAMP_NTZ'),
    ('NOTES', 'VARCHAR'),
])

AUDIT_SCHEMA = StructType([
    StructField('RUN_ID', StringType(), False),
    StructField('SERVICE', StringType(), False),
    StructField('YEAR', IntegerType(), False),
    StructField('MONTH', IntegerType(), False),
    StructField('SOURCE_PATH', StringType(), True),
    StructField('STATUS', StringType(), False),
    StructField('ROWS_READ', LongType(), False),
    StructField('ROWS_VALID', LongType(), False),
    StructField('ROWS_REJECTED', LongType(), False),
    StructField('ROWS_DEDUPED', LongType(), False),
    StructField('ROWS_WRITTEN', LongType(), False),
    StructField('BAD_KEY_ROWS', LongType(), False),
    StructField('BAD_DISTANCE_ROWS', LongType(), False),
    StructField('BAD_DURATION_ROWS', LongType(), False),
    StructField('OUTSIDE_SOURCE_MONTH_ROWS', LongType(), False),
    StructField('EVENT_TIMESTAMP_UTC', TimestampType(), False),
    StructField('NOTES', StringType(), True),
])


def ddl_columns(column_types: OrderedDict[str, str]) -> str:
    return ',\n    '.join(f'{column_name} {snowflake_type}' for column_name, snowflake_type in column_types.items())


def ensure_snowflake_table(table_name: str, column_types: OrderedDict[str, str]) -> None:
    table_fqn = fq_table(table_name)
    sf_run(f'CREATE TABLE IF NOT EXISTS {table_fqn} (\n    {ddl_columns(column_types)}\n)')
    for column_name, snowflake_type in column_types.items():
        sf_run(f'ALTER TABLE {table_fqn} ADD COLUMN IF NOT EXISTS {column_name} {snowflake_type}')


sf_run(f'CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}')
for raw_table in ('YELLOW_TRIPS', 'GREEN_TRIPS'):
    ensure_snowflake_table(raw_table, RAW_COLUMN_TYPES)
ensure_snowflake_table('LOAD_AUDIT', AUDIT_COLUMN_TYPES)
log_step('RAW and audit tables are ready.')


[2026-04-07 22:16:07Z] RAW and audit tables are ready.


## 4. Lectura Parquet

La funcion de lectura construye la ruta fuente desde variables de entorno (`YELLOW_BASE_URL`, `GREEN_BASE_URL`) y descarga a `LOCAL_PARQUET_DIR`. `app.read.parquet(...)` define el DataFrame; el procesamiento real ocurre despues, cuando se ejecuta una accion Spark.


In [4]:
def build_source_url(service: str, year: int, month: int) -> str:
    return f'{BASE_URLS[service]}/{service}_tripdata_{year}-{month:02d}.parquet'


def local_parquet_path(service: str, year: int, month: int) -> Path:
    return LOCAL_PARQUET_DIR / f'{service}_tripdata_{year}-{month:02d}.parquet'


def download_parquet(service: str, year: int, month: int) -> tuple[str, Path]:
    source_url = build_source_url(service, year, month)
    target_path = local_parquet_path(service, year, month)

    if not target_path.exists() or target_path.stat().st_size == 0:
        log_step(f'Downloading {source_url} -> {target_path}')
        urlretrieve(source_url, target_path)
    else:
        log_step(f'Using cached parquet: {target_path}')

    return source_url, target_path


## 5. Estandarizacion y metadatos

Se seleccionan columnas explicitas, se castean tipos esperados y se agregan metadatos RAW: `RUN_ID`, `INGESTED_AT_UTC`, `SOURCE_YEAR`, `SOURCE_MONTH`, `SERVICE_TYPE` y `SOURCE_PATH`. La clave `TRIP_NK` es un hash SHA-256 sobre `SERVICE_TYPE + VENDORID + pickup_datetime + dropoff_datetime + PULOCATIONID + DOLOCATIONID + TRIP_DISTANCE + TOTAL_AMOUNT`; `TRIP_DISTANCE` y `TOTAL_AMOUNT` se incluyen para reducir colisiones porque el dataset no trae un identificador de viaje.


In [5]:
COLUMN_SPECS = [
    ('VENDORID', IntegerType(), ['VendorID', 'vendorid']),
    ('TPEP_PICKUP_DATETIME', TimestampType(), ['tpep_pickup_datetime']),
    ('TPEP_DROPOFF_DATETIME', TimestampType(), ['tpep_dropoff_datetime']),
    ('LPEP_PICKUP_DATETIME', TimestampType(), ['lpep_pickup_datetime']),
    ('LPEP_DROPOFF_DATETIME', TimestampType(), ['lpep_dropoff_datetime']),
    ('STORE_AND_FWD_FLAG', StringType(), ['store_and_fwd_flag', 'Store_and_fwd_flag']),
    ('RATECODEID', IntegerType(), ['RatecodeID', 'ratecodeid']),
    ('PULOCATIONID', IntegerType(), ['PULocationID', 'pulocationid', 'pu_location_id']),
    ('DOLOCATIONID', IntegerType(), ['DOLocationID', 'dolocationid', 'do_location_id']),
    ('PASSENGER_COUNT', DoubleType(), ['passenger_count']),
    ('TRIP_DISTANCE', DoubleType(), ['trip_distance']),
    ('FARE_AMOUNT', DoubleType(), ['fare_amount']),
    ('EXTRA', DoubleType(), ['extra']),
    ('MTA_TAX', DoubleType(), ['mta_tax']),
    ('TIP_AMOUNT', DoubleType(), ['tip_amount']),
    ('TOLLS_AMOUNT', DoubleType(), ['tolls_amount']),
    ('EHAIL_FEE', DoubleType(), ['ehail_fee']),
    ('IMPROVEMENT_SURCHARGE', DoubleType(), ['improvement_surcharge']),
    ('TOTAL_AMOUNT', DoubleType(), ['total_amount']),
    ('PAYMENT_TYPE', IntegerType(), ['payment_type']),
    ('TRIP_TYPE', IntegerType(), ['trip_type']),
    ('CONGESTION_SURCHARGE', DoubleType(), ['congestion_surcharge']),
    ('AIRPORT_FEE', DoubleType(), ['airport_fee']),
]


def resolve_source_column(df, aliases: list[str]):
    columns_by_lower = {column.lower(): column for column in df.columns}
    for alias in aliases:
        source_name = columns_by_lower.get(alias.lower())
        if source_name:
            return F.col(f'`{source_name}`')
    return None


def standardize_raw_columns(df):
    select_exprs = []
    for target_name, target_type, aliases in COLUMN_SPECS:
        source_expr = resolve_source_column(df, aliases)
        if source_expr is None:
            source_expr = F.lit(None)
        select_exprs.append(source_expr.cast(target_type).alias(target_name))
    return df.select(*select_exprs)


def normalized_timestamp_string(column_expr):
    return F.date_format(column_expr, 'yyyy-MM-dd HH:mm:ss.SSSSSS')


def add_metadata_and_key(df, service: str, year: int, month: int, source_path: str):
    pickup_ts = F.coalesce(F.col('TPEP_PICKUP_DATETIME'), F.col('LPEP_PICKUP_DATETIME'))
    dropoff_ts = F.coalesce(F.col('TPEP_DROPOFF_DATETIME'), F.col('LPEP_DROPOFF_DATETIME'))

    required_key_present = (
        F.col('VENDORID').isNotNull()
        & pickup_ts.isNotNull()
        & dropoff_ts.isNotNull()
        & F.col('PULOCATIONID').isNotNull()
        & F.col('DOLOCATIONID').isNotNull()
    )

    natural_key_expr = F.concat_ws(
        '||',
        F.lit(service),
        F.col('VENDORID').cast('string'),
        normalized_timestamp_string(pickup_ts),
        normalized_timestamp_string(dropoff_ts),
        F.col('PULOCATIONID').cast('string'),
        F.col('DOLOCATIONID').cast('string'),
        F.coalesce(F.col('TRIP_DISTANCE').cast('string'), F.lit('__NULL__')),
        F.coalesce(F.col('TOTAL_AMOUNT').cast('string'), F.lit('__NULL__')),
    )

    return (
        df
        .withColumn('RUN_ID', F.lit(RUN_ID))
        .withColumn('SERVICE_TYPE', F.lit(service))
        .withColumn('SOURCE_YEAR', F.lit(year).cast(IntegerType()))
        .withColumn('SOURCE_MONTH', F.lit(month).cast(IntegerType()))
        .withColumn('INGESTED_AT_UTC', F.current_timestamp())
        .withColumn('SOURCE_PATH', F.lit(source_path))
        .withColumn('_PICKUP_TS', pickup_ts)
        .withColumn('_DROPOFF_TS', dropoff_ts)
        .withColumn('TRIP_NK', F.when(required_key_present, F.sha2(natural_key_expr, 256)).otherwise(F.lit(None)))
    )


## 6. Validaciones basicas y deduplicacion

Se rechazan filas sin clave natural, con distancia negativa/nula, duracion negativa/nula o pickup fuera del mes fuente. Luego se deduplica el lote por `TRIP_NK` antes de enviarlo a Snowflake.


In [6]:
QUALITY_COLUMNS = [
    '_PICKUP_TS',
    '_DROPOFF_TS',
    '_DURATION_SECONDS',
    '_KEY_NOT_NULL',
    '_DISTANCE_VALID',
    '_DURATION_VALID',
    '_SOURCE_MONTH_MATCH',
    '_IS_VALID',
]


def source_month_bounds(year: int, month: int) -> tuple[str, str]:
    start = dt.datetime(year, month, 1)
    if month == 12:
        end = dt.datetime(year + 1, 1, 1)
    else:
        end = dt.datetime(year, month + 1, 1)
    return start.strftime('%Y-%m-%d %H:%M:%S'), end.strftime('%Y-%m-%d %H:%M:%S')


def add_quality_flags(df, year: int, month: int):
    source_start, source_end = source_month_bounds(year, month)
    duration_seconds = F.col('_DROPOFF_TS').cast('long') - F.col('_PICKUP_TS').cast('long')

    return (
        df
        .withColumn('_DURATION_SECONDS', duration_seconds)
        .withColumn('_KEY_NOT_NULL', F.col('TRIP_NK').isNotNull())
        .withColumn('_DISTANCE_VALID', F.col('TRIP_DISTANCE').isNotNull() & (F.col('TRIP_DISTANCE') >= F.lit(0.0)))
        .withColumn('_DURATION_VALID', F.col('_DURATION_SECONDS').isNotNull() & (F.col('_DURATION_SECONDS') >= F.lit(0)))
        .withColumn(
            '_SOURCE_MONTH_MATCH',
            (F.col('_PICKUP_TS') >= F.to_timestamp(F.lit(source_start)))
            & (F.col('_PICKUP_TS') < F.to_timestamp(F.lit(source_end)))
        )
        .withColumn(
            '_IS_VALID',
            F.col('_KEY_NOT_NULL')
            & F.col('_DISTANCE_VALID')
            & F.col('_DURATION_VALID')
            & F.col('_SOURCE_MONTH_MATCH')
        )
    )


def bool_failure_count(column_name: str, alias_name: str):
    return F.sum(F.when(F.col(column_name), F.lit(0)).otherwise(F.lit(1))).cast(LongType()).alias(alias_name)


def collect_quality_metrics(df_quality) -> dict[str, int]:
    row = df_quality.agg(
        F.count(F.lit(1)).cast(LongType()).alias('ROWS_READ'),
        F.sum(F.when(F.col('_IS_VALID'), F.lit(1)).otherwise(F.lit(0))).cast(LongType()).alias('ROWS_VALID'),
        bool_failure_count('_KEY_NOT_NULL', 'BAD_KEY_ROWS'),
        bool_failure_count('_DISTANCE_VALID', 'BAD_DISTANCE_ROWS'),
        bool_failure_count('_DURATION_VALID', 'BAD_DURATION_ROWS'),
        bool_failure_count('_SOURCE_MONTH_MATCH', 'OUTSIDE_SOURCE_MONTH_ROWS'),
    ).collect()[0].asDict()

    metrics = {key: int(value or 0) for key, value in row.items()}
    metrics['ROWS_REJECTED'] = metrics['ROWS_READ'] - metrics['ROWS_VALID']
    return metrics


def valid_deduped_batch(df_quality):
    return (
        df_quality
        .filter(F.col('_IS_VALID'))
        .select(*RAW_COLUMNS)
        .dropDuplicates(['TRIP_NK'])
    )


## 7. Escritura idempotente a Snowflake RAW

La carga escribe primero una tabla staging por lote y despues ejecuta `MERGE` contra `RAW.<SERVICE>_TRIPS` usando `TRIP_NK` como clave. Este patron evita `SELECT *`, separa transformacion de publicacion y permite reintentos sin duplicar filas.


In [7]:
def stage_table_name(service: str, year: int, month: int) -> str:
    safe_run_id = re.sub(r'[^A-Za-z0-9_]', '_', RUN_ID).upper()[:80]
    return clean_identifier(f'STG_{service.upper()}_TRIPS_{safe_run_id}_{year}_{month:02d}', 'staging table')


def build_merge_sql(target_table: str, staging_table: str) -> str:
    target_fqn = fq_table(target_table)
    staging_fqn = fq_table(staging_table)
    non_key_columns = [column for column in RAW_COLUMNS if column != 'TRIP_NK']
    update_clause = ',\n    '.join(f'{column} = s.{column}' for column in non_key_columns)
    insert_columns = ', '.join(RAW_COLUMNS)
    insert_values = ', '.join(f's.{column}' for column in RAW_COLUMNS)

    return f"""
MERGE INTO {target_fqn} t
USING {staging_fqn} s
ON t.TRIP_NK = s.TRIP_NK
WHEN MATCHED THEN UPDATE SET
    {update_clause}
WHEN NOT MATCHED THEN INSERT ({insert_columns})
VALUES ({insert_values})
"""


def write_batch_to_raw(df_deduped, service: str, year: int, month: int) -> str:
    target_table = f'{service.upper()}_TRIPS'
    staging_table = stage_table_name(service, year, month)
    staging_fqn = fq_table(staging_table)

    ensure_snowflake_table(target_table, RAW_COLUMN_TYPES)
    sf_run(f'DROP TABLE IF EXISTS {staging_fqn}')

    log_step(f'Writing staging table {staging_fqn}')
    (
        df_deduped.write
        .format('snowflake')
        .options(**SF_OPTIONS)
        .option('dbtable', staging_fqn)
        .mode('overwrite')
        .save()
    )

    log_step(f'Merging staging table into {fq_table(target_table)}')
    sf_run(build_merge_sql(target_table, staging_table))
    sf_run(f'DROP TABLE IF EXISTS {staging_fqn}')
    return staging_table


## 8. Auditoria por lote

Cada lote registra conteos de filas leidas, validas, rechazadas, deduplicadas y escritas en `RAW.LOAD_AUDIT`. Estos conteos son la evidencia de auditoria del run.


In [8]:
def write_audit_row(
    service: str,
    year: int,
    month: int,
    source_path: str,
    status: str,
    metrics: dict[str, int] | None = None,
    notes: str | None = None,
) -> None:
    metrics = metrics or {}
    audit_row = [(
        RUN_ID,
        service,
        year,
        month,
        source_path,
        status,
        int(metrics.get('ROWS_READ', 0)),
        int(metrics.get('ROWS_VALID', 0)),
        int(metrics.get('ROWS_REJECTED', 0)),
        int(metrics.get('ROWS_DEDUPED', 0)),
        int(metrics.get('ROWS_WRITTEN', 0)),
        int(metrics.get('BAD_KEY_ROWS', 0)),
        int(metrics.get('BAD_DISTANCE_ROWS', 0)),
        int(metrics.get('BAD_DURATION_ROWS', 0)),
        int(metrics.get('OUTSIDE_SOURCE_MONTH_ROWS', 0)),
        dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
        notes,
    )]

    audit_df = app.createDataFrame(audit_row, AUDIT_SCHEMA)
    (
        audit_df.write
        .format('snowflake')
        .options(**SF_OPTIONS)
        .option('dbtable', fq_table('LOAD_AUDIT'))
        .mode('append')
        .save()
    )


def print_batch_metrics(service: str, year: int, month: int, metrics: dict[str, int]) -> None:
    log_step(
        f'Batch metrics service={service} period={year}-{month:02d}: '
        f"read={metrics.get('ROWS_READ', 0)}, "
        f"valid={metrics.get('ROWS_VALID', 0)}, "
        f"rejected={metrics.get('ROWS_REJECTED', 0)}, "
        f"deduped={metrics.get('ROWS_DEDUPED', 0)}, "
        f"written={metrics.get('ROWS_WRITTEN', 0)}"
    )


## 9. Ejecucion de la ingesta

Este bloque orquesta todos los lotes. Las acciones Spark principales ocurren en `collect()`, `count()`, `write.save()` y `show()`: ahi se materializan las transformaciones lazy definidas antes.


In [9]:
def is_missing_source_error(error: Exception) -> bool:
    message = str(error)
    return (
        isinstance(error, HTTPError) and error.code in {403, 404}
    ) or 'HTTP Error 403' in message or 'HTTP Error 404' in message


def process_batch(service: str, year: int, month: int) -> tuple[str, str, int, int, int]:
    source_url = build_source_url(service, year, month)
    df_quality = None
    df_deduped = None
    staging_table = stage_table_name(service, year, month)

    try:
        log_step(f'Starting batch service={service} period={year}-{month:02d}')
        source_path, local_path = download_parquet(service, year, month)

        # Ingesta: lectura Parquet. La lectura completa se materializa en acciones posteriores.
        df_raw = app.read.parquet(str(local_path))

        # Estandarizacion y metadatos.
        df_standardized = standardize_raw_columns(df_raw)
        df_enriched = add_metadata_and_key(df_standardized, service, year, month, source_path)

        # Validacion: se persiste en disco para reutilizar el mismo plan en metricas y deduplicacion.
        df_quality = add_quality_flags(df_enriched, year, month).persist(StorageLevel.DISK_ONLY)
        metrics = collect_quality_metrics(df_quality)

        # Deduplicacion por clave natural/proxy.
        df_deduped = valid_deduped_batch(df_quality).persist(StorageLevel.DISK_ONLY)
        rows_written = df_deduped.count()
        metrics['ROWS_WRITTEN'] = rows_written
        metrics['ROWS_DEDUPED'] = metrics['ROWS_VALID'] - rows_written

        print_batch_metrics(service, year, month, metrics)

        if rows_written == 0:
            write_audit_row(service, year, month, source_path, 'NO_VALID_ROWS', metrics, 'No rows passed quality checks.')
            return ('NO_VALID_ROWS', service, year, month, 0)

        # Escritura Snowflake RAW: staging + MERGE idempotente.
        write_batch_to_raw(df_deduped, service, year, month)
        write_audit_row(service, year, month, source_path, 'SUCCESS', metrics)
        log_step(f'Finished batch service={service} period={year}-{month:02d}')
        return ('SUCCESS', service, year, month, rows_written)

    except Exception as error:
        status = 'MISSING' if is_missing_source_error(error) else 'ERROR'
        notes = str(error)[:1000]
        log_step(f'{status} service={service} period={year}-{month:02d}: {notes}')
        try:
            write_audit_row(service, year, month, source_url, status, notes=notes)
        except Exception as audit_error:
            log_step(f'Audit logging failed: {str(audit_error)[:500]}')
        return (status, service, year, month, 0)

    finally:
        if df_quality is not None:
            df_quality.unpersist()
        if df_deduped is not None:
            df_deduped.unpersist()
        try:
            sf_run(f'DROP TABLE IF EXISTS {fq_table(staging_table)}')
        except Exception as cleanup_error:
            log_step(f'Staging cleanup failed for {staging_table}: {str(cleanup_error)[:500]}')


results = []
for service in SERVICES:
    for year in range(START_YEAR, END_YEAR + 1):
        for month in MONTHS:
            results.append(process_batch(service, year, month))

success = sum(1 for status, *_ in results if status == 'SUCCESS')
missing = sum(1 for status, *_ in results if status == 'MISSING')
no_valid = sum(1 for status, *_ in results if status == 'NO_VALID_ROWS')
errors = sum(1 for status, *_ in results if status == 'ERROR')
log_step(f'Run finished: SUCCESS={success}, MISSING={missing}, NO_VALID_ROWS={no_valid}, ERROR={errors}')


[2026-04-07 22:16:07Z] Starting batch service=green period=2020-01
[2026-04-07 22:16:07Z] Downloading https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2020-01.parquet -> /tmp/nyc_taxi_tripdata/green_tripdata_2020-01.parquet
[2026-04-07 22:16:37Z] Batch metrics service=green period=2020-01: read=447770, valid=446880, rejected=890, deduped=138, written=446742
[2026-04-07 22:17:13Z] Writing staging table RAW.STG_GREEN_TRIPS_20260405_TEST_01_2020_01
[2026-04-07 22:17:29Z] Merging staging table into RAW.GREEN_TRIPS
[2026-04-07 22:17:43Z] Finished batch service=green period=2020-01
[2026-04-07 22:17:43Z] Run finished: SUCCESS=1, MISSING=0, NO_VALID_ROWS=0, ERROR=0


## 10. Resumen de auditoria del run

Consulta `RAW.LOAD_AUDIT` para mostrar el resultado de la ejecucion actual por servicio y estado.


In [10]:
df_audit_summary = (
    app.read.format('snowflake')
    .options(**SF_OPTIONS)
    .option('dbtable', fq_table('LOAD_AUDIT'))
    .load()
    .filter(F.col('RUN_ID') == RUN_ID)
)

(
    df_audit_summary
    .groupBy('SERVICE', 'STATUS')
    .agg(
        F.count(F.lit(1)).alias('BATCHES'),
        F.sum('ROWS_READ').alias('ROWS_READ'),
        F.sum('ROWS_REJECTED').alias('ROWS_REJECTED'),
        F.sum('ROWS_DEDUPED').alias('ROWS_DEDUPED'),
        F.sum('ROWS_WRITTEN').alias('ROWS_WRITTEN'),
    )
    .orderBy('SERVICE', 'STATUS')
    .show(truncate=False)
)


+-------+-------+-------+---------+-------------+------------+------------+
|SERVICE|STATUS |BATCHES|ROWS_READ|ROWS_REJECTED|ROWS_DEDUPED|ROWS_WRITTEN|
+-------+-------+-------+---------+-------------+------------+------------+
|green  |SUCCESS|1      |447770   |890          |138         |446742      |
+-------+-------+-------+---------+-------------+------------+------------+

